# NB05 — presentation clusters and abstention calibration

**In:** latent posterior
**Out:** `artifacts/cluster_gmm.pkl`, calibrated `tau`
**Gate:** abstention rate on full-view held-out data ≤ 2%
Partial-view abstention must be materially higher.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
ABSTAIN_MAX = 0.02
N_COMPONENTS = 5
SEED = 0
import numpy as np, pandas as pd, pickle
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split


In [ ]:
# Load
post_p = INTERIM / "latent_posterior.parquet"
post = pd.read_parquet(post_p) if post_p.exists() else None


In [ ]:
# Compute
abstain_full = 1.0
abstain_partial = 0.0
tau = np.nan
if post is not None:
    zcols = [c for c in post.columns if c.startswith("z")]
    Z = post[zcols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
    widths = post["width"].to_numpy() if "width" in post.columns else np.ones(len(post))
    tr, te = train_test_split(np.arange(len(Z)), test_size=0.2, random_state=SEED)
    gmm = GaussianMixture(n_components=min(N_COMPONENTS, max(2, len(Z)//20)), random_state=SEED)
    gmm.fit(Z[tr])
    membership = gmm.predict_proba(Z)
    post["cluster"] = membership.argmax(1)
    post["cluster_mass"] = membership.max(1)
    tau = float(np.quantile(widths[te], 0.98))
    abstain_full = float((widths[te] > tau).mean())
    # simulate partial-view: inflate width as PoE would with one view dropped
    width_partial = widths[te] * np.sqrt(3 / 2)  # 3 experts -> 2 experts, prior still present
    abstain_partial = float((width_partial > tau).mean())
    post["tau"] = tau
    post["abstain"] = post["width"] > tau if "width" in post.columns else False
    post.to_parquet(post_p)
    with open(ARTIFACTS / "cluster_gmm.pkl", "wb") as f:
        pickle.dump({"gmm": gmm, "tau": tau, "zcols": zcols}, f)
    print("tau", tau, "full", abstain_full, "partial", abstain_partial)


In [ ]:
# GATE
note = f"partial-view abstention={abstain_partial:.4f} (must be > full)"
if post is None:
    abstain_full, note = 1.0, "missing latent posterior"
gate("NB05", "abstention_rate_full_view", float(abstain_full), ABSTAIN_MAX, direction="lte",
     n=None if post is None else int(len(post)),
     note=note)
if post is not None and abstain_partial <= abstain_full:
    print("WARNING: partial-view abstention is not higher — PoE mask handling may be wrong")


In [ ]:
# Figures
try:
    import matplotlib.pyplot as plt
    if post is not None:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.hist(post["width"], bins=30)
        ax.axvline(tau, c="red", label="tau")
        ax.set_title("Posterior width and abstention tau")
        ax.legend(); fig.tight_layout()
        fig.savefig(FIGURES / "NB05_width.png", dpi=140)
except Exception as e:
    print(e)
